In [1]:
import os, sys

# Works both in Google Colab and locally.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Update this to wherever you've placed the cloned repo in your Drive.
    PROJECT_ROOT = '/content/drive/MyDrive/thesis_code'
except ImportError:
    # Running locally (e.g. `jupyter lab` from notebooks/): project root is
    # one directory up.
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

os.chdir(PROJECT_ROOT)
sys.path.append(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Access granted to: /content/drive/MyDrive/henrique_code2
📄 Files: ['3D_daily_sentiment_avg_Mag_7_hm.csv', '3D_daily_sentiment_avg_Tariff_hm.csv', 'main.py', 'data_loader_sum.py', 'data_loader_88_sum.py', 'data_loader_88.py', 'data_loader_88_article.py', 'data_loader_article.py', 'spy_2012_2025.csv', 'KXINX_trades_range_vs_sp.csv', 'main.ipynb', '3D_daily_sentiment_avg_articles_new_version_hm.csv', '3D_articles_new_version_hm.csv', '3D_articles_Mag_7_hm.csv', '3D_articles_Tariff_hm.csv', 'vadar_results.csv', 'config.py', 'config_price_only.json', 'config_price_sent.json', 'config_price_sent_mag7.json', 'config_price_sent_tariffs.json', 'config_price_sent_kalshi.json', 'config_price_sent_allexckalshi.json', 'config_price_sent_all.json', 'model.py', 'keras_tuner', 'vadar_plots', 'vadar_plots_cola', '__pycache__', 'config_price_sent_allexc_kalshi.json', 'vadar_

In [2]:
!pip install -q -U keras-tuner


In [3]:
# Cell 1: core imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error
import os

# Swap the loader below to try a different sentiment aggregation/windowing
# variant (see data_loaders/README.md for what each one does):
# from data_loaders.daily_avg_windowed import get_datasets
# from data_loaders.daily_sum import get_datasets
# from data_loaders.daily_sum_windowed import get_datasets
# from data_loaders.article_level import get_datasets
# from data_loaders.article_level_windowed import get_datasets
from data_loaders.daily_avg import get_datasets
from model import (
    build_price,
    build_ft,
    build_model_spy_sentiment,
    build_model_mag7,
    build_model_tariffs,
    build_model_kalshi,
    build_model_allexc_kalshi,
    build_model_all)
from config import cfg



In [4]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

# Define gpus here
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus)

if tf.test.gpu_device_name():
    print("✅ GPU is being used:", tf.test.gpu_device_name())
else:
    print("❌ GPU not in use — training will happen on CPU")

# This now works because 'gpus' is defined
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)


TensorFlow version: 2.18.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ GPU is being used: /device:GPU:0


In [5]:
# ─── New CSV helpers ─────────────────────────────────────────────────────────
def setup_metric_csv(csv_path: str, initial_rows: list):
    if not os.path.isfile(csv_path):
        df = pd.DataFrame(index=initial_rows, columns=[])
        df.index.name = 'Metric'
        df.to_csv(csv_path)
        print(f"Initialized {csv_path} with rows {initial_rows}")

def upsert_experiment(csv_path: str,
                      exp_name: str,
                      row_values: dict,
                      overwrite: bool = False):
    df = pd.read_csv(csv_path, index_col='Metric')

    # ensure the column exists
    if exp_name not in df.columns:
        df[exp_name] = np.nan

    # ensure every metric is a row
    for metric in row_values:
        if metric not in df.index:
            df.loc[metric] = np.nan

    # now write or only fill blanks
    for metric, value in row_values.items():
        current = df.at[metric, exp_name]
        if pd.isna(current) or overwrite:
            df.at[metric, exp_name] = value
        else:
            # skip because there's already a value and overwrite=False
            print(f"→ skipping {metric} (already has {current:.3f})")

    df.to_csv(csv_path)
    print(f"Upserted experiment '{exp_name}' (overwrite={overwrite})")


In [6]:
# ─── Config ─────────────────────────────────────────────────────────────────
CSV_FILE = 'results/article.csv'
OUTPUT_DIR = 'results/article'

# seed exactly the rows you want
setup_metric_csv(CSV_FILE, initial_rows=[
    'Price-only','Kalshi','SPY','Mag7','Tariffs',
    'KalshiInput','AllExcKalshi','All'
])

EXP_NAME   = 'article_sent'   # this becomes your new column name

# make sure your folder exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

## SPY Sentiment Only

In [8]:
import os
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import LSTM, Dropout, Dense, Concatenate, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error
import keras_tuner as kt

from model import build_price, build_ft, build_model_spy_sentiment

MODEL_NAME   = 'SPY'
CONFIG_FILE  = "config_price_sent.json"

# ─────────────────────────────────────────────────────
# Custom tuner that also searches over batch_size & epochs
# ─────────────────────────────────────────────────────
class MyRandomSearch(kt.RandomSearch):
    def run_trial(self, trial, *args, **kwargs):
        hp = trial.hyperparameters

        # 1) Add batch_size and epochs to the search space
        tuned_bsz = hp.Int("batch_size", 8, 64, step=8)   # 8,16,24,…,64
        tuned_eps = hp.Int("epochs",     5, 50, step=5)   # 5,10,15,…,50

        # 2) Override kwargs for model.fit()
        kwargs["batch_size"] = tuned_bsz
        kwargs["epochs"]     = tuned_eps

        # 3) Delegate the rest to RandomSearch
        return super().run_trial(trial, *args, **kwargs)


# ─────────────────────────────────────────────────────
# 1) LOAD DATASETS
# ─────────────────────────────────────────────────────
(ds_pre_tr_spy, ds_pre_val_spy,
 ds_ft_spy,    ds_val_spy,
 Xp_test_spy,  Xs_test_spy,
 y_true_spy,   test_dates_spy,
 price_scaler_spy,
 kalshi_df_spy) = get_datasets(
    cfg,
    use_sentiment=True,
    use_mag=False,
    use_tariff=False,
    use_kalshi=False
)


# ─────────────────────────────────────────────────────
# 2) LOAD SAVED PRICE-ONLY CONFIG and TRAIN PRICE-ONLY
# ─────────────────────────────────────────────────────
with open("config_price_only.json", "r") as f:
    price_cfg = json.load(f)

cfg["lstm_units"] = price_cfg["lstm_units"]
cfg["dropout"]    = price_cfg["dropout"]
cfg["lr"]         = price_cfg["lr"]

price_model_spy = build_price()
price_model_spy.compile(Adam(cfg["lr"]), loss="mse")
price_model_spy.fit(
    ds_pre_tr_spy,
    validation_data=ds_pre_val_spy,
    epochs=cfg["pre_epochs"],
    verbose=1
)


# ─────────────────────────────────────────────────────
# 3) PRICE+SENTIMENT TUNING OR LOAD
# ─────────────────────────────────────────────────────
if Xs_test_spy is not None:
    if os.path.exists(CONFIG_FILE):
        # Load saved fine-tune config
        with open(CONFIG_FILE, "r") as f:
            sent_cfg = json.load(f)

        # Always override architecture‐related keys:
        cfg["dropout"]   = sent_cfg.get("dropout", cfg["dropout"])
        cfg["dropout_s"] = sent_cfg.get("dropout_s", cfg.get("dropout_s", 0.1)) # Provide a default value for dropout_s
        cfg["lr_ft"]     = sent_cfg.get("lr_ft", cfg["lr_ft"])

        # Fall back safely if batch_size/epochs are missing:
        tuned_bsz = sent_cfg.get("batch_size", cfg["batch_size"])
        tuned_eps = sent_cfg.get("epochs",     cfg["ft_epochs"])
    else:
        # Run tuner and save config
        shutil.rmtree('keras_tuner/tune_price_sent', ignore_errors=True)

        tuner_spy = MyRandomSearch(
            build_model_spy_sentiment,
            objective='val_loss',
            max_trials=10,
            executions_per_trial=1,
            directory='keras_tuner',
            project_name='tune_price_sent'
        )

        # Extract fine-tune arrays (price branch inputs, sentiment branch inputs, labels)
        Xp_ft = np.concatenate([x[0] for x, _ in ds_ft_spy], axis=0)
        Xs_ft = np.concatenate([x[1] for x, _ in ds_ft_spy], axis=0)
        y_ft  = np.concatenate([y for _, y in ds_ft_spy], axis=0)

        # Kick off search (no explicit batch_size/epochs here; MyRandomSearch picks them)
        tuner_spy.search(
            [Xp_ft, Xs_ft], y_ft,
            validation_split=0.2
        )

        best_hp_spy = tuner_spy.get_best_hyperparameters(1)[0]
        cfg["dropout"]   = best_hp_spy["dropout"]
        cfg["dropout_s"] = best_hp_spy["dropout_s"]
        cfg["lr_ft"]     = best_hp_spy["learning_rate"]
        tuned_bsz       = best_hp_spy.get("batch_size")
        tuned_eps       = best_hp_spy.get("epochs")

        print("Best batch_size  =", tuned_bsz)
        print("Best epochs      =", tuned_eps)
        print("Best dropout     =", cfg["dropout"])
        print("Best dropout_s   =", cfg["dropout_s"])
        print("Best learning_rate =", cfg["lr_ft"])


        # Write out everything—included tuned batch_size and epochs
        with open(CONFIG_FILE, "w") as f:
            json.dump({
                "lstm_units":  cfg["lstm_units"],
                "dropout":     cfg["dropout"],
                "dropout_s":   cfg["dropout_s"],
                "lr_ft":       cfg["lr_ft"],
                "batch_size":  tuned_bsz,
                "epochs":      tuned_eps
            }, f, indent=4)

    # ─────────────────────────────────────────────────
    # 4) TRAIN PRICE+SENTIMENT MODEL (using tuned values)
    # ─────────────────────────────────────────────────
    # Copy the tuned dropout_s into dropout_rate for build_ft
    cfg["dropout_rate"] = cfg["dropout_s"]

    ft_model_spy = build_ft(Xs_test_spy.shape[2])
    ft_model_spy.get_layer("ps").set_weights(
        price_model_spy.get_layer("ps").get_weights()
    )
    ft_model_spy.get_layer("ps").trainable = False

    ft_model_spy.compile(Adam(cfg["lr_ft"]), loss="mse")
    ft_model_spy.fit(
        ds_ft_spy,
        validation_data=ds_val_spy,
        batch_size=tuned_bsz,
        epochs=tuned_eps // 2,   # split training into two halves if desired
        verbose=1
    )
    ft_model_spy.get_layer("ps").trainable = True
    ft_model_spy.compile(Adam(cfg["lr_ft"]), loss="mse")
    ft_model_spy.fit(
        ds_ft_spy,
        validation_data=ds_val_spy,
        batch_size=tuned_bsz,
        epochs=tuned_eps // 2,
        verbose=1
    )
else:
    ft_model_spy = None


# ─────────────────────────────────────────────────────
# 5) EVALUATE
# ─────────────────────────────────────────────────────
y_ps_spy = np.exp(
    price_scaler_spy.inverse_transform(
        price_model_spy.predict(Xp_test_spy).reshape(-1, 1)
    )
).flatten()

if ft_model_spy is not None:
    y_p_spy = np.exp(
        price_scaler_spy.inverse_transform(
            ft_model_spy.predict([Xp_test_spy, Xs_test_spy]).reshape(-1, 1)
        )
    ).flatten()
else:
    y_p_spy = None

mae_price_spy = mean_absolute_error(y_true_spy, y_ps_spy)
print("Price-only MAE:", mae_price_spy)

if ft_model_spy is not None:
    mae_sent_spy = mean_absolute_error(y_true_spy, y_p_spy)
    print("Price+Sent MAE:", mae_sent_spy)
else:
    mae_sent_spy = np.nan


# ─────────────────────────────────────────────────────
# 6) KALSHI BASELINE (aligned)
# ─────────────────────────────────────────────────────
if kalshi_df_spy is not None:
    kalshi_raw     = kalshi_df_spy["kalshi_price"]
    kalshi_aligned = kalshi_raw.reindex(test_dates_spy, method="ffill")
    assert len(kalshi_aligned) == len(y_true_spy)
    mae_kalshi = mean_absolute_error(y_true_spy, kalshi_aligned.values)
    print("Kalshi MAE (aligned):", mae_kalshi)
else:
    mae_kalshi = np.nan


# ─────────────────────────────────────────────────────
# 7) WRITE TO CSV
# ─────────────────────────────────────────────────────
upsert_experiment(
    CSV_FILE,
    exp_name=EXP_NAME,
    row_values={
        'Price-only': mae_price_spy,
        'Kalshi':     mae_kalshi,
        'SPY':        mae_sent_spy
    },
    overwrite=True
)


# ─────────────────────────────────────────────────────
# 8) PLOT & SAVE
# ─────────────────────────────────────────────────────
fig_spy, ax_spy = plt.subplots(figsize=(12,5))
ax_spy.plot(test_dates_spy, y_true_spy, label="True",        lw=2)
ax_spy.plot(test_dates_spy, y_ps_spy,   label="Price-only")
if ft_model_spy is not None:
    ax_spy.plot(test_dates_spy, y_p_spy, label="Price+Sent")
if kalshi_df_spy is not None:
    ax_spy.plot(test_dates_spy, kalshi_aligned, label="Kalshi")

ax_spy.set_title(f"{MODEL_NAME} Predictions")
ax_spy.legend()
plt.xticks(rotation=45)
plt.tight_layout()
fig_spy.savefig(
    os.path.join(OUTPUT_DIR, f"{MODEL_NAME}.png"),
    bbox_inches='tight'
)
print(f"Saved plot to {os.path.join(OUTPUT_DIR, MODEL_NAME + '.png')}")

→ get_datasets: start
→ get_datasets: loading SPY
  → load_spy()
   SPY post-train shape: (1251, 1)
→ joining daily sentiment
  → load_sentiment()
→ get_datasets: loading raw Kalshi for baseline
  → load_kalshi() (optimized)
→ scaling features
→ making sequences
→ splitting pretrain/fine-tune
→ creating tf.data datasets
→ preparing test data
→ get_datasets: done


ValueError: too many values to unpack (expected 10)

## Mag 7

In [9]:
import os
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error
import keras_tuner as kt

from model import build_price, build_ft, build_model_mag7

MODEL_NAME  = 'Mag7'
CONFIG_FILE = "config_price_sent_mag7.json"

# ─────────────────────────────────────────────────────
# Custom tuner that also searches over batch_size & epochs
# ─────────────────────────────────────────────────────
class MyRandomSearch(kt.RandomSearch):
    def run_trial(self, trial, *args, **kwargs):
        hp = trial.hyperparameters

        # 1) Add batch_size and epochs to the search space
        tuned_bsz = hp.Int("batch_size", 8, 64, step=8)   # 8,16,24,…,64
        tuned_eps = hp.Int("epochs",     5, 50, step=5)   # 5,10,15,…,50

        # 2) Override kwargs for model.fit()
        kwargs["batch_size"] = tuned_bsz
        kwargs["epochs"]     = tuned_eps

        # 3) Delegate the rest to RandomSearch
        return super().run_trial(trial, *args, **kwargs)


# ─────────────────────────────────────────────────────
# 1) LOAD DATASETS (Mag7 features on)
# ─────────────────────────────────────────────────────
(ds_pre_tr_mag7, ds_pre_val_mag7,
 ds_ft_mag7,    ds_val_mag7,
 Xp_test_mag7,  Xs_test_mag7,
 y_true_mag7,   test_dates_mag7,
 price_scaler_mag7,
 kalshi_df_mag7) = get_datasets(
    cfg,
    use_sentiment=False,
    use_mag=True,
    use_tariff=False,
    use_kalshi=False
)


# ─────────────────────────────────────────────────────
# 2) LOAD SAVED PRICE-ONLY CONFIG and TRAIN PRICE-ONLY
# ─────────────────────────────────────────────────────
with open("config_price_only.json", "r") as f:
    price_cfg = json.load(f)

cfg["lstm_units"] = price_cfg["lstm_units"]
cfg["dropout"]    = price_cfg["dropout"]
cfg["lr"]         = price_cfg["lr"]

price_model_mag7 = build_price()
price_model_mag7.compile(Adam(cfg["lr"]), loss="mse")
price_model_mag7.fit(
    ds_pre_tr_mag7,
    validation_data=ds_pre_val_mag7,
    epochs=cfg["pre_epochs"],
    verbose=1
)


# ─────────────────────────────────────────────────────
# 3) PRICE+MAG7 TUNING OR LOAD
# ─────────────────────────────────────────────────────
if Xs_test_mag7 is not None:
    if os.path.exists(CONFIG_FILE):
        # Load saved fine-tune config
        with open(CONFIG_FILE, "r") as f:
            mag7_cfg = json.load(f)

        # Override architecture-related keys:
        cfg["dropout"]   = mag7_cfg.get("dropout", cfg["dropout"])
        cfg["dropout_s"] = mag7_cfg.get("dropout_s", cfg["dropout_s"])
        cfg["lr_ft"]     = mag7_cfg.get("lr_ft", cfg["lr_ft"])

        # Fall back safely if batch_size/epochs are missing:
        tuned_bsz = mag7_cfg.get("batch_size", cfg["batch_size"])
        tuned_eps = mag7_cfg.get("epochs",     cfg["ft_epochs"])
    else:
        # Run tuner and save config
        shutil.rmtree('keras_tuner/tune_mag7', ignore_errors=True)

        # s_dims = number of Mag7 features per time step
        s_dims = Xs_test_mag7.shape[2]

        tuner_mag7 = MyRandomSearch(
            lambda hp: build_model_mag7(hp, s_dims),
            objective='val_loss',
            max_trials=10,
            executions_per_trial=1,
            directory='keras_tuner',
            project_name='tune_mag7'
        )

        # Extract fine-tune arrays (price inputs, Mag7 inputs, labels)
        Xp_ft_mag7 = np.concatenate([x[0] for x, _ in ds_ft_mag7], axis=0)
        Xs_ft_mag7 = np.concatenate([x[1] for x, _ in ds_ft_mag7], axis=0)
        y_ft_mag7  = np.concatenate([y for _, y in ds_ft_mag7], axis=0)

        # Kick off search (MyRandomSearch picks batch_size & epochs)
        tuner_mag7.search(
            [Xp_ft_mag7, Xs_ft_mag7], y_ft_mag7,
            validation_split=0.2
        )

        best_hp_mag7 = tuner_mag7.get_best_hyperparameters(1)[0]
        cfg["dropout"]   = best_hp_mag7["dropout"]
        cfg["dropout_s"] = best_hp_mag7["dropout_s"]
        cfg["lr_ft"]     = best_hp_mag7["learning_rate"]
        tuned_bsz       = best_hp_mag7.get("batch_size")
        tuned_eps       = best_hp_mag7.get("epochs")

        # Write out everything—included tuned batch_size & epochs
        with open(CONFIG_FILE, "w") as f:
            json.dump({
                "lstm_units":  cfg["lstm_units"],
                "dropout":     cfg["dropout"],
                "dropout_s":   cfg["dropout_s"],
                "lr_ft":       cfg["lr_ft"],
                "batch_size":  tuned_bsz,
                "epochs":      tuned_eps
            }, f, indent=4)

    # ─────────────────────────────────────────────────
    # 4) TRAIN PRICE+MAG7 MODEL (using tuned values)
    # ─────────────────────────────────────────────────
    # Copy tuned dropout_s into dropout_rate before calling build_ft()
    cfg["dropout_rate"] = cfg["dropout_s"]

    ft_model_mag7 = build_ft(Xs_test_mag7.shape[2])
    ft_model_mag7.get_layer("ps").set_weights(
        price_model_mag7.get_layer("ps").get_weights()
    )
    ft_model_mag7.get_layer("ps").trainable = False

    ft_model_mag7.compile(Adam(cfg["lr_ft"]), loss="mse")
    ft_model_mag7.fit(
        ds_ft_mag7,
        validation_data=ds_val_mag7,
        batch_size=tuned_bsz,
        epochs=tuned_eps // 2,   # split into two halves if you prefer
        verbose=1
    )
    ft_model_mag7.get_layer("ps").trainable = True
    ft_model_mag7.compile(Adam(cfg["lr_ft"]), loss="mse")
    ft_model_mag7.fit(
        ds_ft_mag7,
        validation_data=ds_val_mag7,
        batch_size=tuned_bsz,
        epochs=tuned_eps // 2,
        verbose=1
    )
else:
    ft_model_mag7 = None


# ─────────────────────────────────────────────────────
# 5) EVALUATE
# ─────────────────────────────────────────────────────
y_ps_mag7 = np.exp(
    price_scaler_mag7.inverse_transform(
        price_model_mag7.predict(Xp_test_mag7).reshape(-1, 1)
    )
).flatten()

if ft_model_mag7 is not None:
    y_pa_mag7 = np.exp(
        price_scaler_mag7.inverse_transform(
            ft_model_mag7.predict([Xp_test_mag7, Xs_test_mag7]).reshape(-1, 1)
        )
    ).flatten()
else:
    y_pa_mag7 = None

mae_price_mag7   = mean_absolute_error(y_true_mag7, y_ps_mag7)
mae_allsent_mag7 = (mean_absolute_error(y_true_mag7, y_pa_mag7)
                    if y_pa_mag7 is not None else np.nan)
print("Mag7 Price-only MAE:", mae_price_mag7)
print("Mag7 Price+Mag7 MAE:", mae_allsent_mag7)


# ─────────────────────────────────────────────────────
# 6) KALSHI BASELINE (aligned)
# ─────────────────────────────────────────────────────
if kalshi_df_mag7 is not None:
    kalshi_raw_mag7     = kalshi_df_mag7["kalshi_price"]
    kalshi_aligned_mag7 = kalshi_raw_mag7.reindex(test_dates_mag7, method="ffill")
    assert len(kalshi_aligned_mag7) == len(y_true_mag7)
    mae_kalshi_mag7     = mean_absolute_error(y_true_mag7, kalshi_aligned_mag7.values)
    print("Mag7 Kalshi MAE (aligned):", mae_kalshi_mag7)
else:
    mae_kalshi_mag7 = np.nan


# ─────────────────────────────────────────────────────
# 7) WRITE TO CSV
# ─────────────────────────────────────────────────────
upsert_experiment(
    CSV_FILE,
    exp_name=EXP_NAME,
    row_values={
        'Price-only': mae_price_mag7,
        'Kalshi':     mae_kalshi_mag7,
        'Mag7':       mae_allsent_mag7
    },
    overwrite=True
)


# ─────────────────────────────────────────────────────
# 8) PLOT & SAVE
# ─────────────────────────────────────────────────────
fig_mag7, ax_mag7 = plt.subplots(figsize=(12,5))
ax_mag7.plot(test_dates_mag7, y_true_mag7,           label="True", lw=2)
ax_mag7.plot(test_dates_mag7, y_pa_mag7,             label="Price+Mag7")
if kalshi_df_mag7 is not None:
    ax_mag7.plot(test_dates_mag7, kalshi_aligned_mag7, label="Kalshi")

ax_mag7.set_title(f"{MODEL_NAME} Predictions")
ax_mag7.legend()
plt.xticks(rotation=45)
plt.tight_layout()
fig_mag7.savefig(
    os.path.join(OUTPUT_DIR, f"{MODEL_NAME}.png"),
    bbox_inches='tight'
)
print(f"Saved plot to {os.path.join(OUTPUT_DIR, MODEL_NAME + '.png')}")

→ get_datasets: start
→ get_datasets: loading SPY
  → load_spy()
   SPY post-train shape: (1251, 1)
→ joining Mag-7 sentiment
  → load_mag()
→ get_datasets: loading raw Kalshi for baseline
  → load_kalshi() (optimized)
→ scaling features
→ making sequences
→ splitting pretrain/fine-tune
→ creating tf.data datasets
→ preparing test data
→ get_datasets: done


ValueError: too many values to unpack (expected 10)

## Tariffs

In [10]:
import os
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error
import keras_tuner as kt

from model import build_price, build_ft, build_model_tariffs

MODEL_NAME  = 'Tariffs'
CONFIG_FILE = "config_price_sent_tariffs.json"

# ─────────────────────────────────────────────────────
# Custom tuner that also searches over batch_size & epochs
# ─────────────────────────────────────────────────────
class MyRandomSearch(kt.RandomSearch):
    def run_trial(self, trial, *args, **kwargs):
        hp = trial.hyperparameters

        # 1) Add batch_size and epochs to the search space
        tuned_bsz = hp.Int("batch_size", 8, 64, step=8)   # 8,16,24,…,64
        tuned_eps = hp.Int("epochs",     5, 50, step=5)   # 5,10,15,…,50

        # 2) Override kwargs for model.fit()
        kwargs["batch_size"] = tuned_bsz
        kwargs["epochs"]     = tuned_eps

        # 3) Delegate to RandomSearch
        return super().run_trial(trial, *args, **kwargs)


# ─────────────────────────────────────────────────────
# 1) LOAD DATASETS (Tariff features on)
# ─────────────────────────────────────────────────────
(ds_pre_tr_tariffs, ds_pre_val_tariffs,
 ds_ft_tariffs,    ds_val_tariffs,
 Xp_test_tariffs,  Xs_test_tariffs,
 y_true_tariffs,   test_dates_tariffs,
 price_scaler_tariffs,
 kalshi_df_tariffs) = get_datasets(
    cfg,
    use_sentiment=False,
    use_mag=False,
    use_tariff=True,
    use_kalshi=False
)


# ─────────────────────────────────────────────────────
# 2) LOAD SAVED PRICE-ONLY CONFIG and TRAIN PRICE-ONLY
# ─────────────────────────────────────────────────────
with open("config_price_only.json", "r") as f:
    price_cfg = json.load(f)

cfg["lstm_units"] = price_cfg["lstm_units"]
cfg["dropout"]    = price_cfg["dropout"]
cfg["lr"]         = price_cfg["lr"]

price_model_tariffs = build_price()
price_model_tariffs.compile(Adam(cfg["lr"]), loss="mse")
price_model_tariffs.fit(
    ds_pre_tr_tariffs,
    validation_data=ds_pre_val_tariffs,
    epochs=cfg["pre_epochs"],
    verbose=1
)


# ─────────────────────────────────────────────────────
# 3) PRICE+TARIFF TUNING OR LOAD
# ─────────────────────────────────────────────────────
if Xs_test_tariffs is not None:
    if os.path.exists(CONFIG_FILE):
        # Load saved fine-tune config
        with open(CONFIG_FILE, "r") as f:
            tariff_cfg = json.load(f)

        # Override architecture-related keys:
        cfg["dropout"]   = tariff_cfg.get("dropout", cfg["dropout"])
        cfg["dropout_s"] = tariff_cfg.get("dropout_s", cfg["dropout_s"])
        cfg["lr_ft"]     = tariff_cfg.get("lr_ft", cfg["lr_ft"])

        # Fall back safely if batch_size/epochs are missing:
        tuned_bsz = tariff_cfg.get("batch_size", cfg["batch_size"])
        tuned_eps = tariff_cfg.get("epochs",     cfg["ft_epochs"])
    else:
        # Run tuner and save config
        shutil.rmtree('keras_tuner/tune_tariffs', ignore_errors=True)

        # s_dims = number of tariff features per time step
        s_dims = Xs_test_tariffs.shape[2]

        tuner_tariffs = MyRandomSearch(
            lambda hp: build_model_tariffs(hp, s_dims),
            objective='val_loss',
            max_trials=10,
            executions_per_trial=1,
            directory='keras_tuner',
            project_name='tune_tariffs'
        )

        # Extract fine-tune arrays (price inputs, tariff inputs, labels)
        Xp_ft_tariffs = np.concatenate([x[0] for x, _ in ds_ft_tariffs], axis=0)
        Xs_ft_tariffs = np.concatenate([x[1] for x, _ in ds_ft_tariffs], axis=0)
        y_ft_tariffs  = np.concatenate([y for _, y in ds_ft_tariffs], axis=0)

        # Kick off search (MyRandomSearch picks batch_size & epochs)
        tuner_tariffs.search(
            [Xp_ft_tariffs, Xs_ft_tariffs], y_ft_tariffs,
            validation_split=0.2
        )

        best_hp_tariffs = tuner_tariffs.get_best_hyperparameters(1)[0]
        cfg["dropout"]   = best_hp_tariffs["dropout"]
        cfg["dropout_s"] = best_hp_tariffs["dropout_s"]
        cfg["lr_ft"]     = best_hp_tariffs["learning_rate"]
        tuned_bsz       = best_hp_tariffs.get("batch_size")
        tuned_eps       = best_hp_tariffs.get("epochs")

        # Write out everything—including tuned batch_size & epochs
        with open(CONFIG_FILE, "w") as f:
            json.dump({
                "lstm_units":  cfg["lstm_units"],
                "dropout":     cfg["dropout"],
                "dropout_s":   cfg["dropout_s"],
                "lr_ft":       cfg["lr_ft"],
                "batch_size":  tuned_bsz,
                "epochs":      tuned_eps
            }, f, indent=4)

    # ─────────────────────────────────────────────────
    # 4) TRAIN PRICE+TARIFF MODEL (using tuned values)
    # ─────────────────────────────────────────────────
    # Copy tuned dropout_s into dropout_rate before calling build_ft()
    cfg["dropout_rate"] = cfg["dropout_s"]

    ft_model_tariffs = build_ft(Xs_test_tariffs.shape[2])
    ft_model_tariffs.get_layer("ps").set_weights(
        price_model_tariffs.get_layer("ps").get_weights()
    )
    ft_model_tariffs.get_layer("ps").trainable = False

    ft_model_tariffs.compile(Adam(cfg["lr_ft"]), loss="mse")
    ft_model_tariffs.fit(
        ds_ft_tariffs,
        validation_data=ds_val_tariffs,
        batch_size=tuned_bsz,
        epochs=tuned_eps // 2,   # split into two halves if you prefer
        verbose=1
    )
    ft_model_tariffs.get_layer("ps").trainable = True
    ft_model_tariffs.compile(Adam(cfg["lr_ft"]), loss="mse")
    ft_model_tariffs.fit(
        ds_ft_tariffs,
        validation_data=ds_val_tariffs,
        batch_size=tuned_bsz,
        epochs=tuned_eps // 2,
        verbose=1
    )
else:
    ft_model_tariffs = None


# ─────────────────────────────────────────────────────
# 5) EVALUATE
# ─────────────────────────────────────────────────────
y_ps_tariffs = np.exp(
    price_scaler_tariffs.inverse_transform(
        price_model_tariffs.predict(Xp_test_tariffs).reshape(-1, 1)
    )
).flatten()

if ft_model_tariffs is not None:
    y_pa_tariffs = np.exp(
        price_scaler_tariffs.inverse_transform(
            ft_model_tariffs.predict([Xp_test_tariffs, Xs_test_tariffs]).reshape(-1, 1)
        )
    ).flatten()
else:
    y_pa_tariffs = None

mae_price_tariffs = mean_absolute_error(y_true_tariffs, y_ps_tariffs)
mae_sent_tariffs  = (mean_absolute_error(y_true_tariffs, y_pa_tariffs)
                     if y_pa_tariffs is not None else np.nan)
print("Tariffs Price-only MAE:", mae_price_tariffs)
print("Tariffs Price+Tariff MAE:", mae_sent_tariffs)


# ─────────────────────────────────────────────────────
# 6) KALSHI BASELINE (aligned)
# ─────────────────────────────────────────────────────
if kalshi_df_tariffs is not None:
    kalshi_raw_tariffs     = kalshi_df_tariffs["kalshi_price"]
    kalshi_aligned_tariffs = kalshi_raw_tariffs.reindex(test_dates_tariffs, method="ffill")
    assert len(kalshi_aligned_tariffs) == len(y_true_tariffs)
    mae_kalshi_tariffs     = mean_absolute_error(y_true_tariffs, kalshi_aligned_tariffs.values)
    print("Tariffs Kalshi MAE (aligned):", mae_kalshi_tariffs)
else:
    mae_kalshi_tariffs = np.nan


# ─────────────────────────────────────────────────────
# 7) WRITE TO CSV
# ─────────────────────────────────────────────────────
upsert_experiment(
    CSV_FILE,
    exp_name=EXP_NAME,
    row_values={
        'Price-only': mae_price_tariffs,
        'Kalshi':     mae_kalshi_tariffs,
        'Tariffs':    mae_sent_tariffs
    },
    overwrite=True
)


# ─────────────────────────────────────────────────────
# 8) PLOT & SAVE
# ─────────────────────────────────────────────────────
fig_tariffs, ax_tariffs = plt.subplots(figsize=(12,5))
ax_tariffs.plot(test_dates_tariffs, y_true_tariffs,           label="True", lw=2)
ax_tariffs.plot(test_dates_tariffs, y_pa_tariffs,             label="Price+Tariff")
if kalshi_df_tariffs is not None:
    ax_tariffs.plot(test_dates_tariffs, kalshi_aligned_tariffs, label="Kalshi")

ax_tariffs.set_title(f"{MODEL_NAME} Predictions")
ax_tariffs.legend()
plt.xticks(rotation=45)
plt.tight_layout()
fig_tariffs.savefig(
    os.path.join(OUTPUT_DIR, f"{MODEL_NAME}.png"),
    bbox_inches='tight'
)
print(f"Saved plot to {os.path.join(OUTPUT_DIR, MODEL_NAME + '.png')}")

→ get_datasets: start
→ get_datasets: loading SPY
  → load_spy()
   SPY post-train shape: (1251, 1)
→ joining Tariff sentiment
  → load_tariff()
→ get_datasets: loading raw Kalshi for baseline
  → load_kalshi() (optimized)
→ scaling features
→ making sequences
→ splitting pretrain/fine-tune
→ creating tf.data datasets
→ preparing test data
→ get_datasets: done


ValueError: too many values to unpack (expected 10)

## Kalshi as input

In [11]:
import os
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error
import keras_tuner as kt

from model import build_price, build_ft, build_model_kalshi

MODEL_NAME  = 'KalshiInput'
CONFIG_FILE = "config_price_sent_kalshi.json"

# ─────────────────────────────────────────────────────
# Custom tuner that also searches over batch_size & epochs
# ─────────────────────────────────────────────────────
class MyRandomSearch(kt.RandomSearch):
    def run_trial(self, trial, *args, **kwargs):
        hp = trial.hyperparameters

        # 1) Add batch_size and epochs to the search space
        tuned_bsz = hp.Int("batch_size", 8, 64, step=8)   # 8,16,24,…,64
        tuned_eps = hp.Int("epochs",     5, 50, step=5)   # 5,10,15,…,50

        # 2) Override kwargs for model.fit()
        kwargs["batch_size"] = tuned_bsz
        kwargs["epochs"]     = tuned_eps

        # 3) Delegate to RandomSearch
        return super().run_trial(trial, *args, **kwargs)


# ─────────────────────────────────────────────────────
# 1) LOAD DATASETS (Kalshi input enabled)
# ─────────────────────────────────────────────────────
(ds_pre_tr_kalshi, ds_pre_val_kalshi,
 ds_ft_kalshi,    ds_val_kalshi,
 Xp_test_kalshi,  Xs_test_kalshi,
 y_true_kalshi,   test_dates_kalshi,
 price_scaler_kalshi,
 kalshi_df_kalshi) = get_datasets(
    cfg,
    use_sentiment=False,
    use_mag=False,
    use_tariff=False,
    use_kalshi=True
)


# ─────────────────────────────────────────────────────
# 2) LOAD SAVED PRICE-ONLY CONFIG and TRAIN PRICE-ONLY
# ─────────────────────────────────────────────────────
with open("config_price_only.json", "r") as f:
    price_cfg = json.load(f)

cfg["lstm_units"] = price_cfg["lstm_units"]
cfg["dropout"]    = price_cfg["dropout"]
cfg["lr"]         = price_cfg["lr"]

price_model_kalshi = build_price()
price_model_kalshi.compile(Adam(cfg["lr"]), loss="mse")
price_model_kalshi.fit(
    ds_pre_tr_kalshi,
    validation_data=ds_pre_val_kalshi,
    epochs=cfg["pre_epochs"],
    verbose=1
)


# ─────────────────────────────────────────────────────
# 3) PRICE+KALSHI TUNING OR LOAD
# ─────────────────────────────────────────────────────
if Xs_test_kalshi is not None:
    if os.path.exists(CONFIG_FILE):
        # Load saved fine-tune config
        with open(CONFIG_FILE, "r") as f:
            kalshi_cfg = json.load(f)

        # Override architecture-related keys:
        cfg["dropout"]   = kalshi_cfg.get("dropout", cfg["dropout"])
        cfg["dropout_s"] = kalshi_cfg.get("dropout_s", cfg["dropout_s"])
        cfg["lr_ft"]     = kalshi_cfg.get("lr_ft", cfg["lr_ft"])

        # Fall back safely if batch_size/epochs are missing:
        tuned_bsz = kalshi_cfg.get("batch_size", cfg["batch_size"])
        tuned_eps = kalshi_cfg.get("epochs",     cfg["ft_epochs"])
    else:
        # Run tuner and save config
        shutil.rmtree('keras_tuner/tune_kalshi', ignore_errors=True)

        # s_dims = number of Kalshi features per time step
        s_dims = Xs_test_kalshi.shape[2]

        tuner_kalshi = MyRandomSearch(
            lambda hp: build_model_kalshi(hp, s_dims),
            objective='val_loss',
            max_trials=10,
            executions_per_trial=1,
            directory='keras_tuner',
            project_name='tune_kalshi'
        )

        # Extract fine-tune arrays (price inputs, Kalshi inputs, labels)
        Xp_ft_kalshi = np.concatenate([x[0] for x, _ in ds_ft_kalshi], axis=0)
        Xs_ft_kalshi = np.concatenate([x[1] for x, _ in ds_ft_kalshi], axis=0)
        y_ft_kalshi  = np.concatenate([y for _, y in ds_ft_kalshi], axis=0)

        # Kick off search (MyRandomSearch picks batch_size & epochs)
        tuner_kalshi.search(
            [Xp_ft_kalshi, Xs_ft_kalshi], y_ft_kalshi,
            validation_split=0.2
        )

        best_hp_kalshi = tuner_kalshi.get_best_hyperparameters(1)[0]
        cfg["dropout"]   = best_hp_kalshi["dropout"]
        cfg["dropout_s"] = best_hp_kalshi["dropout_s"]
        cfg["lr_ft"]     = best_hp_kalshi["learning_rate"]
        tuned_bsz       = best_hp_kalshi.get("batch_size")
        tuned_eps       = best_hp_kalshi.get("epochs")

        # Write out everything—including tuned batch_size & epochs
        with open(CONFIG_FILE, "w") as f:
            json.dump({
                "lstm_units":  cfg["lstm_units"],
                "dropout":     cfg["dropout"],
                "dropout_s":   cfg["dropout_s"],
                "lr_ft":       cfg["lr_ft"],
                "batch_size":  tuned_bsz,
                "epochs":      tuned_eps
            }, f, indent=4)

    # ─────────────────────────────────────────────────
    # 4) TRAIN PRICE+KALSHI MODEL (using tuned values)
    # ─────────────────────────────────────────────────
    # Copy tuned dropout_s into dropout_rate before build_ft()
    cfg["dropout_rate"] = cfg["dropout_s"]

    ft_model_kalshi = build_ft(Xs_test_kalshi.shape[2])
    ft_model_kalshi.get_layer("ps").set_weights(
        price_model_kalshi.get_layer("ps").get_weights()
    )
    ft_model_kalshi.get_layer("ps").trainable = False

    ft_model_kalshi.compile(Adam(cfg["lr_ft"]), loss="mse")
    ft_model_kalshi.fit(
        ds_ft_kalshi,
        validation_data=ds_val_kalshi,
        batch_size=tuned_bsz,
        epochs=tuned_eps // 2,   # split into two halves if desired
        verbose=1
    )
    ft_model_kalshi.get_layer("ps").trainable = True
    ft_model_kalshi.compile(Adam(cfg["lr_ft"]), loss="mse")
    ft_model_kalshi.fit(
        ds_ft_kalshi,
        validation_data=ds_val_kalshi,
        batch_size=tuned_bsz,
        epochs=tuned_eps // 2,
        verbose=1
    )
else:
    ft_model_kalshi = None


# ─────────────────────────────────────────────────────
# 5) EVALUATE
# ─────────────────────────────────────────────────────
y_ps_kalshi = np.exp(
    price_scaler_kalshi.inverse_transform(
        price_model_kalshi.predict(Xp_test_kalshi).reshape(-1, 1)
    )
).flatten()

if ft_model_kalshi is not None:
    y_pa_kalshi = np.exp(
        price_scaler_kalshi.inverse_transform(
            ft_model_kalshi.predict([Xp_test_kalshi, Xs_test_kalshi]).reshape(-1, 1)
        )
    ).flatten()
else:
    y_pa_kalshi = None

mae_price_kalshi    = mean_absolute_error(y_true_kalshi, y_ps_kalshi)
mae_allinput_kalshi = (mean_absolute_error(y_true_kalshi, y_pa_kalshi)
                       if y_pa_kalshi is not None else np.nan)
print("KalshiInput Price-only MAE:", mae_price_kalshi)
print("KalshiInput Price+Kalshi MAE:", mae_allinput_kalshi)


# ─────────────────────────────────────────────────────
# 6) KALSHI BASELINE (aligned)
# ─────────────────────────────────────────────────────
if kalshi_df_kalshi is not None:
    kalshi_raw       = kalshi_df_kalshi["kalshi_price"]
    kalshi_aligned   = kalshi_raw.reindex(test_dates_kalshi, method="ffill")
    assert len(kalshi_aligned) == len(y_true_kalshi)
    mae_kalshi_base  = mean_absolute_error(y_true_kalshi, kalshi_aligned.values)
    print("KalshiInput Kalshi baseline MAE (aligned):", mae_kalshi_base)
else:
    mae_kalshi_base = np.nan


# ─────────────────────────────────────────────────────
# 7) WRITE TO CSV
# ─────────────────────────────────────────────────────
upsert_experiment(
    CSV_FILE,
    exp_name=EXP_NAME,
    row_values={
        'Price-only':  mae_price_kalshi,
        'Kalshi':      mae_kalshi_base,
        'KalshiInput': mae_allinput_kalshi
    },
    overwrite=True
)


# ─────────────────────────────────────────────────────
# 8) PLOT & SAVE
# ─────────────────────────────────────────────────────
fig_kalshi, ax_kalshi = plt.subplots(figsize=(12,5))
ax_kalshi.plot(test_dates_kalshi, y_true_kalshi,      label="True", lw=2)
ax_kalshi.plot(test_dates_kalshi, y_pa_kalshi,        label="Price+Kalshi")
if kalshi_df_kalshi is not None:
    ax_kalshi.plot(test_dates_kalshi, kalshi_aligned,  label="Kalshi")

ax_kalshi.set_title(f"{MODEL_NAME} Predictions")
ax_kalshi.legend()
plt.xticks(rotation=45)
plt.tight_layout()
fig_kalshi.savefig(
    os.path.join(OUTPUT_DIR, f"{MODEL_NAME}.png"),
    bbox_inches='tight'
)
print(f"Saved plot to {os.path.join(OUTPUT_DIR, MODEL_NAME + '.png')}")

→ get_datasets: start
→ get_datasets: loading SPY
  → load_spy()
   SPY post-train shape: (1251, 1)
→ get_datasets: loading raw Kalshi for baseline
  → load_kalshi() (optimized)
→ get_datasets: joining Kalshi as input feature
→ scaling features
→ making sequences
→ splitting pretrain/fine-tune
→ creating tf.data datasets
→ preparing test data
→ get_datasets: done


ValueError: too many values to unpack (expected 10)

## All Sentiment Data (Exc. Kalshi)

In [ ]:
import os
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error
import keras_tuner as kt

from model import build_price, build_ft, build_model_allexc_kalshi

MODEL_NAME  = 'AllExcKalshi'
CONFIG_FILE = "config_price_sent_allexc_kalshi.json"

# ─────────────────────────────────────────────────────
# Custom tuner that also searches over batch_size & epochs
# ─────────────────────────────────────────────────────
class MyRandomSearch(kt.RandomSearch):
    def run_trial(self, trial, *args, **kwargs):
        hp = trial.hyperparameters

        # 1) Add batch_size and epochs to the search space
        tuned_bsz = hp.Int("batch_size", 8, 64, step=8)   # 8,16,24,…,64
        tuned_eps = hp.Int("epochs",     5, 50, step=5)   # 5,10,15,…,50

        # 2) Override kwargs for model.fit()
        kwargs["batch_size"] = tuned_bsz
        kwargs["epochs"]     = tuned_eps

        # 3) Delegate to RandomSearch
        return super().run_trial(trial, *args, **kwargs)


# ─────────────────────────────────────────────────────
# 1) LOAD DATASETS (all sentiment features, no Kalshi input)
# ─────────────────────────────────────────────────────
(ds_pre_tr_all, ds_pre_val_all,
 ds_ft_all,    ds_val_all,
 Xp_test_all,  Xs_test_all,
 y_true_all,   test_dates_all,
 price_scaler_all, kalshi_df_all) = get_datasets(
    cfg,
    use_sentiment=True,
    use_mag=True,
    use_tariff=True,
    use_kalshi=False
)


# ─────────────────────────────────────────────────────
# 2) LOAD SAVED PRICE-ONLY CONFIG and TRAIN PRICE-ONLY
# ─────────────────────────────────────────────────────
with open("config_price_only.json", "r") as f:
    price_cfg = json.load(f)

cfg["lstm_units"] = price_cfg["lstm_units"]
cfg["dropout"]    = price_cfg["dropout"]
cfg["lr"]         = price_cfg["lr"]

price_model_all = build_price()
price_model_all.compile(Adam(cfg["lr"]), loss="mse")
price_model_all.fit(
    ds_pre_tr_all,
    validation_data=ds_pre_val_all,
    epochs=cfg["pre_epochs"],
    verbose=1
)


# ─────────────────────────────────────────────────────
# 3) PRICE+ALL-SENTIMENT TUNING OR LOAD
# ─────────────────────────────────────────────────────
if Xs_test_all is not None:
    if os.path.exists(CONFIG_FILE):
        # Load saved fine-tune config
        with open(CONFIG_FILE, "r") as f:
            all_cfg = json.load(f)

        # Override architecture-related keys:
        cfg["dropout"]   = all_cfg.get("dropout", cfg["dropout"])
        cfg["dropout_s"] = all_cfg.get("dropout_s", cfg["dropout_s"])
        cfg["lr_ft"]     = all_cfg.get("lr_ft", cfg["lr_ft"])

        # Fall back safely if batch_size/epochs are missing:
        tuned_bsz = all_cfg.get("batch_size", cfg["batch_size"])
        tuned_eps = all_cfg.get("epochs",     cfg["ft_epochs"])
    else:
        # Run tuner and save config
        shutil.rmtree('keras_tuner/tune_allexc_kalshi', ignore_errors=True)

        # s_dims = number of combined sentiment/mag/tariff features per time step
        s_dims = Xs_test_all.shape[2]

        tuner_all = MyRandomSearch(
            lambda hp: build_model_allexc_kalshi(hp, s_dims),
            objective='val_loss',
            max_trials=10,
            executions_per_trial=1,
            directory='keras_tuner',
            project_name='tune_allexc_kalshi'
        )

        # Extract fine-tune arrays (price inputs, all-sentiment inputs, labels)
        Xp_ft_all = np.concatenate([x[0] for x, _ in ds_ft_all], axis=0)
        Xs_ft_all = np.concatenate([x[1] for x, _ in ds_ft_all], axis=0)
        y_ft_all  = np.concatenate([y for _, y in ds_ft_all], axis=0)

        # Kick off search (MyRandomSearch picks batch_size & epochs)
        tuner_all.search(
            [Xp_ft_all, Xs_ft_all], y_ft_all,
            validation_split=0.2
        )

        best_hp_all = tuner_all.get_best_hyperparameters(1)[0]
        cfg["dropout"]   = best_hp_all["dropout"]
        cfg["dropout_s"] = best_hp_all["dropout_s"]
        cfg["lr_ft"]     = best_hp_all["learning_rate"]
        tuned_bsz       = best_hp_all.get("batch_size")
        tuned_eps       = best_hp_all.get("epochs")

        # Write out everything—including tuned batch_size & epochs
        with open(CONFIG_FILE, "w") as f:
            json.dump({
                "lstm_units":  cfg["lstm_units"],
                "dropout":     cfg["dropout"],
                "dropout_s":   cfg["dropout_s"],
                "lr_ft":       cfg["lr_ft"],
                "batch_size":  tuned_bsz,
                "epochs":      tuned_eps
            }, f, indent=4)

    # ─────────────────────────────────────────────────
    # 4) TRAIN PRICE+ALL-SENTIMENT MODEL (using tuned values)
    # ─────────────────────────────────────────────────
    # Copy tuned dropout_s into dropout_rate before build_ft()
    cfg["dropout_rate"] = cfg["dropout_s"]

    ft_model_all = build_ft(Xs_test_all.shape[2])
    ft_model_all.get_layer("ps").set_weights(
        price_model_all.get_layer("ps").get_weights()
    )
    ft_model_all.get_layer("ps").trainable = False

    ft_model_all.compile(Adam(cfg["lr_ft"]), loss="mse")
    ft_model_all.fit(
        ds_ft_all,
        validation_data=ds_val_all,
        batch_size=tuned_bsz,
        epochs=tuned_eps // 2,   # split into two halves if you prefer
        verbose=1
    )
    ft_model_all.get_layer("ps").trainable = True
    ft_model_all.compile(Adam(cfg["lr_ft"]), loss="mse")
    ft_model_all.fit(
        ds_ft_all,
        validation_data=ds_val_all,
        batch_size=tuned_bsz,
        epochs=tuned_eps // 2,
        verbose=1
    )
else:
    ft_model_all = None


# ─────────────────────────────────────────────────────
# 5) EVALUATE
# ─────────────────────────────────────────────────────
y_ps_all = np.exp(
    price_scaler_all.inverse_transform(
        price_model_all.predict(Xp_test_all).reshape(-1, 1)
    )
).flatten()

if ft_model_all is not None:
    y_pa_all = np.exp(
        price_scaler_all.inverse_transform(
            ft_model_all.predict([Xp_test_all, Xs_test_all]).reshape(-1, 1)
        )
    ).flatten()
else:
    y_pa_all = None

mae_price_all   = mean_absolute_error(y_true_all, y_ps_all)
mae_allsent_all = (mean_absolute_error(y_true_all, y_pa_all)
                   if y_pa_all is not None else np.nan)
print("AllExcKalshi Price-only MAE:", mae_price_all)
print("AllExcKalshi Price+Sent MAE:", mae_allsent_all)


# ─────────────────────────────────────────────────────
# 6) KALSHI BASELINE (aligned)  — note: using baseline Kalshi price only if available
# ─────────────────────────────────────────────────────
if kalshi_df_all is not None:
    kalshi_raw_all     = kalshi_df_all["kalshi_price"]
    kalshi_aligned_all = kalshi_raw_all.reindex(test_dates_all, method="ffill")
    assert len(kalshi_aligned_all) == len(y_true_all)
    mae_kalshi_all     = mean_absolute_error(y_true_all, kalshi_aligned_all.values)
    print("AllExcKalshi Kalshi baseline MAE (aligned):", mae_kalshi_all)
else:
    mae_kalshi_all = np.nan


# ─────────────────────────────────────────────────────
# 7) WRITE TO CSV
# ─────────────────────────────────────────────────────
upsert_experiment(
    CSV_FILE,
    exp_name=EXP_NAME,
    row_values={
        'Price-only':   mae_price_all,
        'Kalshi':       mae_kalshi_all,
        'AllExcKalshi': mae_allsent_all
    },
    overwrite=True
)


# ─────────────────────────────────────────────────────
# 8) PLOT & SAVE
# ─────────────────────────────────────────────────────
fig_all, ax_all = plt.subplots(figsize=(12,5))
ax_all.plot(test_dates_all, y_true_all,           label="True",        lw=2)
ax_all.plot(test_dates_all, y_pa_all,             label="Price+Sent")
if kalshi_df_all is not None:
    ax_all.plot(test_dates_all, kalshi_aligned_all, label="Kalshi")

ax_all.set_title(f"{MODEL_NAME} Predictions")
ax_all.legend()
plt.xticks(rotation=45)
plt.tight_layout()
fig_all.savefig(
    os.path.join(OUTPUT_DIR, f"{MODEL_NAME}.png"),
    bbox_inches='tight'
)
print(f"Saved plot to {os.path.join(OUTPUT_DIR, MODEL_NAME + '.png')}")

## All Sentiment (Inc. Kalshi)

In [ ]:
import os
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error
import keras_tuner as kt

from model import build_price, build_ft, build_model_all

MODEL_NAME  = 'All'
CONFIG_FILE = "config_price_sent_all.json"

# ─────────────────────────────────────────────────────
# Custom tuner that also searches over batch_size & epochs
# ─────────────────────────────────────────────────────
class MyRandomSearch(kt.RandomSearch):
    def run_trial(self, trial, *args, **kwargs):
        hp = trial.hyperparameters

        # 1) Add batch_size and epochs to the search space
        tuned_bsz = hp.Int("batch_size", 8, 64, step=8)   # 8,16,24,…,64
        tuned_eps = hp.Int("epochs",     5, 50, step=5)   # 5,10,15,…,50

        # 2) Override kwargs for model.fit()
        kwargs["batch_size"] = tuned_bsz
        kwargs["epochs"]     = tuned_eps

        # 3) Delegate to RandomSearch
        super().run_trial(trial, *args, **kwargs)


# ─────────────────────────────────────────────────────
# 1) LOAD DATASETS (all features including Kalshi)
# ─────────────────────────────────────────────────────
(ds_pre_tr_all, ds_pre_val_all,
 ds_ft_all,    ds_val_all,
 Xp_test_all,  Xs_test_all,
 y_true_all,   test_dates_all,
 price_scaler_all, kalshi_df_all) = get_datasets(
    cfg,
    use_sentiment=True,
    use_mag=True,
    use_tariff=True,
    use_kalshi=True
)


# ─────────────────────────────────────────────────────
# 2) LOAD SAVED PRICE-ONLY CONFIG and TRAIN PRICE-ONLY
# ─────────────────────────────────────────────────────
with open("config_price_only.json", "r") as f:
    price_cfg = json.load(f)

cfg["lstm_units"] = price_cfg["lstm_units"]
cfg["dropout"]    = price_cfg["dropout"]
cfg["lr"]         = price_cfg["lr"]

price_model_all = build_price()
price_model_all.compile(Adam(cfg["lr"]), loss="mse")
price_model_all.fit(
    ds_pre_tr_all,
    validation_data=ds_pre_val_all,
    epochs=cfg["pre_epochs"],
    verbose=1
)


# ─────────────────────────────────────────────────────
# 3) PRICE+ALL-INPUTS TUNING OR LOAD
# ─────────────────────────────────────────────────────
if Xs_test_all is not None:
    if os.path.exists(CONFIG_FILE):
        # Load saved fine-tune config
        with open(CONFIG_FILE, "r") as f:
            all_cfg = json.load(f)

        # Override architecture-related keys:
        cfg["dropout"]   = all_cfg.get("dropout", cfg["dropout"])
        cfg["dropout_s"] = all_cfg.get("dropout_s", cfg["dropout_s"])
        cfg["lr_ft"]     = all_cfg.get("lr_ft", cfg["lr_ft"])

        # Fall back safely if batch_size/epochs are missing:
        tuned_bsz = all_cfg.get("batch_size", cfg["batch_size"])
        tuned_eps = all_cfg.get("epochs",     cfg["ft_epochs"])
    else:
        # Run tuner and save config
        shutil.rmtree('keras_tuner/tune_all', ignore_errors=True)

        # s_dims = number of combined features (sentiment+mag+tariff+kalshi) per time step
        s_dims = Xs_test_all.shape[2]

        tuner_all = MyRandomSearch(
            lambda hp: build_model_all(hp, s_dims),
            objective='val_loss',
            max_trials=10,
            executions_per_trial=1,
            directory='keras_tuner',
            project_name='tune_all'
        )

        # Extract fine-tune arrays (price inputs, all‐features inputs, labels)
        Xp_ft_all = np.concatenate([x[0] for x, _ in ds_ft_all], axis=0)
        Xs_ft_all = np.concatenate([x[1] for x, _ in ds_ft_all], axis=0)
        y_ft_all  = np.concatenate([y for _, y in ds_ft_all], axis=0)

        # Kick off search (MyRandomSearch picks batch_size & epochs)
        tuner_all.search(
            [Xp_ft_all, Xs_ft_all], y_ft_all,
            validation_split=0.2
        )

        best_hp_all = tuner_all.get_best_hyperparameters(1)[0]
        cfg["dropout"]   = best_hp_all["dropout"]
        cfg["dropout_s"] = best_hp_all["dropout_s"]
        cfg["lr_ft"]     = best_hp_all["learning_rate"]
        tuned_bsz       = best_hp_all.get("batch_size")
        tuned_eps       = best_hp_all.get("epochs")

        # Write out everything—including tuned batch_size & epochs
        with open(CONFIG_FILE, "w") as f:
            json.dump({
                "lstm_units":  cfg["lstm_units"],
                "dropout":     cfg["dropout"],
                "dropout_s":   cfg["dropout_s"],
                "lr_ft":       cfg["lr_ft"],
                "batch_size":  tuned_bsz,
                "epochs":      tuned_eps
            }, f, indent=4)

    # ─────────────────────────────────────────────────
    # 4) TRAIN PRICE+ALL-INPUTS MODEL (using tuned values)
    # ─────────────────────────────────────────────────
    # Copy tuned dropout_s into dropout_rate before build_ft()
    cfg["dropout_rate"] = cfg["dropout_s"]

    ft_model_all = build_ft(Xs_test_all.shape[2])
    ft_model_all.get_layer("ps").set_weights(
        price_model_all.get_layer("ps").get_weights()
    )
    ft_model_all.get_layer("ps").trainable = False

    ft_model_all.compile(Adam(cfg["lr_ft"]), loss="mse")
    ft_model_all.fit(
        ds_ft_all,
        validation_data=ds_val_all,
        batch_size=tuned_bsz,
        epochs=tuned_eps // 2,   # split into two halves if you prefer
        verbose=1
    )
    ft_model_all.get_layer("ps").trainable = True
    ft_model_all.compile(Adam(cfg["lr_ft"]), loss="mse")
    ft_model_all.fit(
        ds_ft_all,
        validation_data=ds_val_all,
        batch_size=tuned_bsz,
        epochs=tuned_eps // 2,
        verbose=1
    )
else:
    ft_model_all = None


# ─────────────────────────────────────────────────────
# 5) EVALUATE
# ─────────────────────────────────────────────────────
y_ps_all = np.exp(
    price_scaler_all.inverse_transform(
        price_model_all.predict(Xp_test_all).reshape(-1, 1)
    )
).flatten()

if ft_model_all is not None:
    y_pa_all = np.exp(
        price_scaler_all.inverse_transform(
            ft_model_all.predict([Xp_test_all, Xs_test_all]).reshape(-1, 1)
        )
    ).flatten()
else:
    y_pa_all = None

mae_price_all     = mean_absolute_error(y_true_all, y_ps_all)
mae_allinputs_all = (mean_absolute_error(y_true_all, y_pa_all)
                     if y_pa_all is not None else np.nan)
print("All Price-only MAE:", mae_price_all)
print("All Price+AllInputs MAE:", mae_allinputs_all)


# ─────────────────────────────────────────────────────
# 6) KALSHI BASELINE (aligned)
# ─────────────────────────────────────────────────────
if kalshi_df_all is not None:
    kalshi_raw_all     = kalshi_df_all["kalshi_price"]
    kalshi_aligned_all = kalshi_raw_all.reindex(test_dates_all, method="ffill")
    assert len(kalshi_aligned_all) == len(y_true_all)
    mae_kalshi_all     = mean_absolute_error(y_true_all, kalshi_aligned_all.values)
    print("All Kalshi baseline MAE (aligned):", mae_kalshi_all)
else:
    mae_kalshi_all = np.nan


# ─────────────────────────────────────────────────────
# 7) WRITE TO CSV
# ─────────────────────────────────────────────────────
upsert_experiment(
    CSV_FILE,
    exp_name=EXP_NAME,
    row_values={
        'Price-only': mae_price_all,
        'Kalshi':     mae_kalshi_all,
        'All':        mae_allinputs_all
    },
    overwrite=True
)


# ─────────────────────────────────────────────────────
# 8) PLOT & SAVE
# ─────────────────────────────────────────────────────
fig_all, ax_all = plt.subplots(figsize=(12,5))
ax_all.plot(test_dates_all, y_true_all,           label="True", lw=2)
ax_all.plot(test_dates_all, y_pa_all,             label="Price+AllInputs")
if kalshi_df_all is not None:
    ax_all.plot(test_dates_all, kalshi_aligned_all, label="Kalshi")

ax_all.set_title(f"{MODEL_NAME} Predictions")
ax_all.legend()
plt.xticks(rotation=45)
plt.tight_layout()
fig_all.savefig(
    os.path.join(OUTPUT_DIR, f"{MODEL_NAME}.png"),
    bbox_inches='tight'
)
print(f"Saved plot to {os.path.join(OUTPUT_DIR, MODEL_NAME + '.png')}")

## Plotting

In [ ]:
# 1) Gather your Price+Sent MAEs
mae_price_sent = {
    'SPY':           mae_sent_spy,
    'Mag7':          mae_allsent_mag7,
    'Tariffs':       mae_sent_tariffs,
    'KalshiInput':   mae_allinput_kalshi,
    'AllExcKalshi':  mae_allsent_all,
    'All':           mae_allinputs_all
}

# 2) Pick the best model
best_model = min(mae_price_sent, key=mae_price_sent.get)
print(f"Best Price+Sent model: {best_model} (MAE={mae_price_sent[best_model]:.4f})")

# 3) Wire up the corresponding variables
if best_model == 'SPY':
    dates = test_dates_spy
    true  = y_true_spy
    price = y_ps_spy
    sent  = y_p_spy
    kalshi = pd.Series(
        kalshi_raw.values,
        index=kalshi_raw.index
    ) if 'kalshi_raw' in locals() else None

elif best_model == 'Mag7':
    dates = test_dates_mag7
    true  = y_true_mag7
    price = y_ps_mag7
    sent  = y_pa_mag7
    kalshi = pd.Series(
        kts_mag7,
        index=test_dates_mag7[:len(kts_mag7)]
    ) if 'kts_mag7' in locals() else None

elif best_model == 'Tariffs':
    dates = test_dates_tariffs
    true  = y_true_tariffs
    price = y_ps_tariffs
    sent  = y_pa_tariffs
    kalshi = pd.Series(
        kts_tariffs,
        index=test_dates_tariffs[:len(kts_tariffs)]
    ) if 'kts_tariffs' in locals() else None

elif best_model == 'KalshiInput':
    dates = test_dates_kalshi
    true  = y_true_kalshi
    price = y_ps_kalshi
    sent  = y_pa_kalshi
    kalshi = pd.Series(
        kts_kalshi,
        index=test_dates_kalshi[:len(kts_kalshi)]
    ) if 'kts_kalshi' in locals() else None

elif best_model == 'AllExcKalshi':
    dates = test_dates_all
    true  = y_true_all
    price = y_ps_all
    sent  = y_pa_all
    # no Kalshi input in this block
    kalshi = None

else:  # 'All'
    dates = test_dates_all
    true  = y_true_all
    price = y_ps_all
    sent  = y_pa_all
    kalshi = pd.Series(
        kts_all,
        index=test_dates_all[:len(kts_all)]
    ) if 'kts_all' in locals() else None

# 4) Plot everything
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(dates, true,  label='True',        lw=2)
ax.plot(dates, price, label='Price-only')
ax.plot(dates, sent,  label=f'Price+Sent ({best_model})')

if kalshi is not None:
    ax.plot(kalshi.index, kalshi.values, label='Kalshi')

ax.set_title(f'Comparison: {best_model} vs Baselines')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
configs = {}
for name in ["price_only", "price_sent", "price_sent_tariffs", "price_sent_mag7", "price_sent_kalshi", "price_sent_allexckalshi", "price_sent_all"]:
    with open(f"config_{name}.json", "r") as f:
        configs[name] = json.load(f)

from pprint import pprint
pprint(configs)
